In [1]:
!nvidia-smi


Sun Oct  4 04:43:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!git clone https://github.com/Maheendra-mj/3-day-inference-challenge

Cloning into '3-day-inference-challenge'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 60 (delta 10), reused 56 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 47.24 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [5]:
%cd 3-day-inference-challenge

/kaggle/working/3-day-inference-challenge


In [8]:
!bash scripts/kaggle_bootstrap.sh

index, name, memory.total [MiB], driver_version
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for infer-lab (pyproje

In [9]:
!infer-lab bench --config configs/smoke.yaml --tag smoke


2026-10-03 16:54:55,259 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-03 16:54:55,433 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 2.40MB/s]
tokenizer_config.json: 7.30kB [00:00, 17.2MB/s]
vocab.json: 2.78MB [00:00, 44.1MB/s]
merges.txt: 1.67MB [00:00, 87.1MB/s]
tokenizer.json: 7.03MB [00:00, 117MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 988M/988M [00:03<00:00, 285MB/s]
Loading weights: 100%|█| 290/290 [00:00<00:00, 462.89it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 242/242 [00:00<00:00, 738kB/s]
2026-10-03 16:55:11,594 INFO infer_lab.bench.runner: run 1  bs=1   pl=64    ol=32    tok/s=    30.3  ttft_p50=   35.8ms  tpot_p50= 32.94ms  util= 39%  power=33.2W  s

In [10]:
!infer-lab bench --config configs/bench.yaml --tag baseline


2026-10-03 16:55:34,463 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-03 16:55:34,646 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 555.27it/s, Materializing param=m
2026-10-03 16:55:56,449 INFO infer_lab.bench.runner: run 3  bs=1   pl=128   ol=128   tok/s=    28.9  ttft_p50=   39.4ms  tpot_p50= 34.46ms  util= 39%  power=33.2W  sm_clk= 699MHz
2026-10-03 16:56:13,316 INFO infer_lab.bench.runner: run 4  bs=1   pl=512   ol=128   tok/s=    30.7  ttft_p50=   49.6ms  tpot_p50= 31.84ms  util= 35%  power=37.4W  sm_clk=1124MHz
2026-10-03 16:56:30,466 INFO infer_lab.bench.runner: run 5  bs=4   pl=128   ol=128   tok/s=   119.6  ttft_p50=   37.6ms  tpot_p50= 33.34ms  util= 36%  power=36.0W  sm_clk=1072MHz
2026-10-03 16:56:47,944 INFO infe

In [11]:
!infer-lab kernels


vector_add  torch         n=67108864     3.281 ms    245.4 GB/s  ( 76.7% of T4 peak)
vector_add  scalar        n=67108864     3.578 ms    225.1 GB/s  ( 70.3% of T4 peak)
vector_add  vec4          n=67108864     3.379 ms    238.4 GB/s  ( 74.5% of T4 peak)
matmul      torch_cublas  n=512          0.082 ms   3.283 TFLOPS  ( 40.5% of T4 peak)
matmul      naive         n=512          0.454 ms   0.591 TFLOPS  (  7.3% of T4 peak)
matmul      tiled16       n=512          0.293 ms   0.917 TFLOPS  ( 11.3% of T4 peak)
matmul      torch_cublas  n=1024         0.383 ms   5.608 TFLOPS  ( 69.2% of T4 peak)
matmul      naive         n=1024         5.128 ms   0.419 TFLOPS  (  5.2% of T4 peak)
matmul      tiled16       n=1024         3.043 ms   0.706 TFLOPS  (  8.7% of T4 peak)
matmul      torch_cublas  n=2048         4.842 ms   3.548 TFLOPS  ( 43.8% of T4 peak)
matmul      naive         n=2048        44.502 ms   0.386 TFLOPS  (  4.8% of T4 peak)
matmul      tiled16       n=2048        29.210 ms   0.588

In [12]:
!infer-lab report


2026-10-02 08:24:50,250 INFO numexpr.utils: NumExpr defaulting to 4 threads.
 id      tag  backend  batch_size  prompt_len  output_len  output_tokens_per_s  ttft_p50_ms  ttft_p99_ms  tpot_p50_ms  e2e_p99_ms
  1    smoke hf_torch           1          64          32                34.25        37.17        37.17        28.92      933.59
  2    smoke hf_torch           4          64          32               132.33        31.79        31.79        30.15      966.56
  3 baseline hf_torch           1         128         128                33.76        33.36        33.66        29.44     3842.15
  4 baseline hf_torch           1         512         128                33.79        46.67        47.11        29.59     3826.54
  5 baseline hf_torch           4         128         128               137.84        32.15        33.54        29.09     3736.30
  6 baseline hf_torch           4         512         128               132.42       175.42       175.77        28.96     3909.17
  7 baseline 

In [12]:
!infer-lab profile --bs 32 --pl 128 512 --ol 32 --trace


2026-10-03 16:58:37,772 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 496.45it/s, Materializing param=m
2026-10-03 16:58:41,196 INFO infer_lab.bench.profile: profiling bs=32 pl=128 ol=32
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
2026-10-03 16:59:25,332 INFO infer_lab.bench.profile: profiling bs=32 pl=512 ol=32

 bs    pl | prefill wall  GPU busy | decode wall/step GPU busy/step GPU idle kernels/step avg kernel | verdict
 32   128 |      251.2ms   482.8ms |          33.03ms       75.95ms       0%         1422     53.4us | GPU-bound
 32   512 |     1446

In [13]:
!cat results/profile/top_ops_bs32_pl512_ol32.txt


----------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
----------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                           decode_step         0.00%       0.000us         0.00%       0.000us       0.000us        1.766s        56.13%        1.766s      56.963ms            31  
                                                               prefill         0.00%       0.000us         0.00%       0.000us       0.000us       

In [14]:
!infer-lab bench --config configs/bench.yaml --tag baseline_v2


2026-10-03 17:00:35,063 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-03 17:00:35,264 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 511.66it/s, Materializing param=m
2026-10-03 17:00:56,773 INFO infer_lab.bench.runner: run 11  bs=1   pl=128   ol=128   tok/s=    29.4  ttft_p50=   37.3ms  tpot_p50= 34.16ms  util= 37%  power=36.4W  sm_clk= 758MHz
2026-10-03 17:01:13,863 INFO infer_lab.bench.runner: run 12  bs=1   pl=512   ol=128   tok/s=    29.4  ttft_p50=   50.0ms  tpot_p50= 34.85ms  util= 34%  power=40.9W  sm_clk=1088MHz
2026-10-03 17:01:30,586 INFO infer_lab.bench.runner: run 13  bs=4   pl=128   ol=128   tok/s=   123.9  ttft_p50=   38.4ms  tpot_p50= 32.33ms  util= 36%  power=41.4W  sm_clk=1148MHz
2026-10-03 17:01:47,890 INFO i

In [15]:
!infer-lab report


 id         tag  backend  batch_size  prompt_len  output_len  output_tokens_per_s  ttft_p50_ms  ttft_p99_ms  tpot_p50_ms  e2e_p99_ms  torch_peak_mem_mb  gpu.gpu0.util_mean_pct  gpu.gpu0.power_mean_w
  1       smoke hf_torch           1          64          32                30.26        35.78        35.78        32.94     1056.89             962.28                   39.00                  33.16
  2       smoke hf_torch           4          64          32               121.09        35.73        35.73        32.92     1056.19             974.24                   38.80                  32.75
  3    baseline hf_torch           1         128         128                28.95        39.42        41.05        34.46     4434.40             966.26                   38.65                  33.17
  4    baseline hf_torch           1         512         128                30.74        49.63        52.36        31.84     4311.96            1008.94                   35.13                  37.43
  5  